# 4 — Geração das nove séries desagregadas no conjunto de teste

Este notebook carrega **um modelo para cada uma das nove configurações**:

| Intervalo da saída | Janelas de entrada |
|---:|:---|
| 1 hora | 3, 5 e 7 dias |
| 2 horas | 3, 5 e 7 dias |
| 4 horas | 3, 5 e 7 dias |

Para cada configuração, ele gera um CSV contínuo do período de teste com duas colunas:

- `data_hora`
- `chuva_mm`

O modelo é executado apenas nos dias com precipitação diária maior que zero. Dias completos com total diário igual a zero são preenchidos com zero em todos os intervalos. Falhas de observação **não são transformadas em zero**: permanecem como valores ausentes para evitar criar falsos dias secos.

As nove séries usam, por padrão, o mesmo período comum de teste. Os totais diários são preservados exatamente pela saída `Softmax` da GRU.


In [1]:
# Se faltar alguma biblioteca, descomente e execute uma vez:
# %pip install torch pandas numpy ipywidgets

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("PyTorch não está instalado. Instale-o antes de executar o notebook.") from e

try:
    from IPython.display import display, clear_output
except ImportError:
    display = print

pd.set_option("display.max_columns", 30)


## Configuração


In [2]:
CAMINHO_DADOS = Path("dados_processados/chuva_horaria_padronizada.csv")
PASTA_MODELOS = Path("modelos")
PASTA_SAIDA = Path("resultados/series_teste_desagregadas")

INTERVALOS_HORAS = (1, 2, 4)
JANELAS_DIAS = (3, 5, 7)
BATCH_SIZE_PREVISAO = 512
TOLERANCIA_DIA_SECO_MM = 1e-9

# True: os nove CSVs começam e terminam nas mesmas datas.
# Isso é recomendado para comparar posteriormente máximas e acumulados.
USAR_PERIODO_COMUM = True

# Limites opcionais dentro do conjunto de teste. Deixe None para usar todo o teste.
DATA_INICIO_TESTE = None  # exemplo: "2021-01-01"
DATA_FIM_TESTE = None     # exemplo: "2023-12-31"

# None escolhe automaticamente, dentro da combinação, o checkpoint com o menor
# MSE de validação. Também é possível escrever o nome exato de cada arquivo .pt.
MODELOS_ESCOLHIDOS = {
    (1, 3): None,
    (1, 5): None,
    (1, 7): None,
    (2, 3): None,
    (2, 5): None,
    (2, 7): None,
    (4, 3): None,
    (4, 5): None,
    (4, 7): None,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)


Dispositivo: cpu


## Arquitetura e leitura dos checkpoints


In [3]:
class GRUDesagregadora(nn.Module):
    def __init__(self, hidden_size, n_outputs, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0 if num_layers == 1 else 0.1,
        )
        self.fc = nn.Linear(hidden_size, n_outputs)

    def forward(self, x, total_central):
        sequencia, h_n = self.gru(x)
        logits = self.fc(sequencia[:, -1, :])
        fracoes = torch.softmax(logits, dim=1)
        previsao = fracoes * total_central[:, None]
        return previsao, fracoes, h_n


def torch_load_compativel(caminho, map_location="cpu"):
    try:
        return torch.load(caminho, map_location=map_location, weights_only=False)
    except TypeError:  # compatibilidade com versões antigas do PyTorch
        return torch.load(caminho, map_location=map_location)


def validar_configuracao(cfg, caminho=None):
    exigidas = {
        "intervalo_horas", "janela_dias", "hidden_size", "num_layers",
        "n_outputs", "media_log_treino", "desvio_log_treino", "fim_validacao",
    }
    faltantes = exigidas - set(cfg)
    if faltantes:
        origem = f" em {caminho}" if caminho is not None else ""
        raise KeyError(f"Configuração incompleta{origem}. Faltam: {sorted(faltantes)}")
    intervalo = int(cfg["intervalo_horas"])
    janela = int(cfg["janela_dias"])
    if intervalo not in INTERVALOS_HORAS or janela not in JANELAS_DIAS:
        raise ValueError(f"Configuração não reconhecida: intervalo={intervalo}, janela={janela}")
    if int(cfg["n_outputs"]) != 24 // intervalo:
        raise ValueError("n_outputs é incompatível com intervalo_horas.")
    if float(cfg["desvio_log_treino"]) <= 0:
        raise ValueError("desvio_log_treino deve ser positivo.")


def carregar_checkpoint(caminho, device=DEVICE):
    caminho = Path(caminho)
    ckpt = torch_load_compativel(caminho, map_location=device)
    if not {"state_dict", "config"}.issubset(ckpt):
        raise KeyError(f"Checkpoint incompatível: {caminho.name}")
    cfg = ckpt["config"]
    validar_configuracao(cfg, caminho)
    modelo = GRUDesagregadora(
        hidden_size=int(cfg["hidden_size"]),
        n_outputs=int(cfg["n_outputs"]),
        num_layers=int(cfg["num_layers"]),
    ).to(device)
    modelo.load_state_dict(ckpt["state_dict"])
    modelo.eval()
    return modelo, cfg


def catalogar_modelos(pasta=PASTA_MODELOS):
    pasta = Path(pasta)
    if not pasta.exists():
        raise FileNotFoundError(f"Pasta de modelos não encontrada: {pasta.resolve()}")
    linhas_catalogo = []
    erros = []
    for caminho in sorted(pasta.glob("*.pt")):
        try:
            ckpt = torch_load_compativel(caminho, map_location="cpu")
            cfg = ckpt["config"]
            validar_configuracao(cfg, caminho)
            mse = cfg.get("melhor_MSE_validacao", np.nan)
            linhas_catalogo.append({
                "arquivo": caminho.name,
                "caminho": str(caminho),
                "intervalo_horas": int(cfg["intervalo_horas"]),
                "janela_dias": int(cfg["janela_dias"]),
                "hidden_size": int(cfg["hidden_size"]),
                "num_layers": int(cfg["num_layers"]),
                "repeticao": cfg.get("nome", caminho.stem).split("_r")[-1],
                "MSE_validacao": float(mse) if pd.notna(mse) else np.nan,
                "fim_validacao": str(cfg["fim_validacao"]),
            })
        except Exception as e:
            erros.append({"arquivo": caminho.name, "erro": str(e)})
    catalogo = pd.DataFrame(linhas_catalogo)
    if catalogo.empty:
        detalhe = f" Erros: {erros[:3]}" if erros else ""
        raise FileNotFoundError(f"Nenhum checkpoint compatível foi encontrado em {pasta.resolve()}.{detalhe}")
    if erros:
        warnings.warn(f"{len(erros)} arquivo(s) .pt incompatível(is) foram ignorados.")
    return catalogo.sort_values(["intervalo_horas", "janela_dias", "MSE_validacao", "arquivo"])


## Catálogo e seleção dos nove modelos


In [4]:
catalogo = catalogar_modelos()
display(catalogo.drop(columns="caminho").reset_index(drop=True))


,arquivo,intervalo_horas,janela_dias,hidden_size,num_layers,repeticao,MSE_validacao,fim_validacao
0,gru_1h_3d_32oc_r5.pt,1,3,32,1,5,3.134438,2024-02-16
1,gru_1h_5d_32oc_r5.pt,1,5,32,1,5,3.236085,2024-02-16
2,gru_1h_7d_32oc_r5.pt,1,7,32,1,5,3.252737,2024-02-16
3,gru_2h_3d_32oc_r5.pt,2,3,32,1,5,8.203114,2024-02-16
4,gru_2h_5d_32oc_r5.pt,2,5,32,1,5,8.518005,2024-02-16
5,gru_2h_7d_32oc_r5.pt,2,7,32,1,5,8.848252,2024-02-16
6,gru_4h_3d_32oc_r5.pt,4,3,32,1,5,19.148885,2024-02-16
7,gru_4h_5d_32oc_r5.pt,4,5,32,1,5,19.994781,2024-02-16
8,gru_4h_7d_32oc_r5.pt,4,7,32,1,5,21.273851,2024-02-16


In [5]:
def resolver_selecao(catalogo, escolhas=MODELOS_ESCOLHIDOS):
    selecionados = {}
    faltantes = []

    for intervalo in INTERVALOS_HORAS:
        for janela in JANELAS_DIAS:
            chave = (intervalo, janela)
            candidatos = catalogo[
                (catalogo["intervalo_horas"] == intervalo)
                & (catalogo["janela_dias"] == janela)
            ].copy()
            if candidatos.empty:
                faltantes.append(chave)
                continue

            escolha = escolhas.get(chave)
            if escolha is None:
                candidatos["ordem_mse"] = candidatos["MSE_validacao"].fillna(np.inf)
                linha = candidatos.sort_values(["ordem_mse", "arquivo"]).iloc[0]
                caminho = Path(linha["caminho"])
            else:
                caminho = Path(escolha)
                if not caminho.exists():
                    caminho = PASTA_MODELOS / caminho.name
                if not caminho.exists():
                    raise FileNotFoundError(f"Modelo escolhido não encontrado: {escolha}")
                linha_match = candidatos[candidatos["arquivo"] == caminho.name]
                if linha_match.empty:
                    raise ValueError(
                        f"{caminho.name} não pertence à configuração {intervalo} h / {janela} dias."
                    )
            selecionados[chave] = caminho

    if faltantes:
        texto = ", ".join(f"{i}h-{j}d" for i, j in faltantes)
        raise ValueError(f"Faltam modelos para: {texto}. Treine essas configurações no notebook 02.")
    return selecionados


selecionados = resolver_selecao(catalogo)
display(pd.DataFrame([
    {"intervalo_horas": i, "janela_dias": j, "modelo": p.name}
    for (i, j), p in selecionados.items()
]))


,intervalo_horas,janela_dias,modelo
0,1,3,gru_1h_3d_32oc_r5.pt
1,1,5,gru_1h_5d_32oc_r5.pt
2,1,7,gru_1h_7d_32oc_r5.pt
3,2,3,gru_2h_3d_32oc_r5.pt
4,2,5,gru_2h_5d_32oc_r5.pt
5,2,7,gru_2h_7d_32oc_r5.pt
6,4,3,gru_4h_3d_32oc_r5.pt
7,4,5,gru_4h_5d_32oc_r5.pt
8,4,7,gru_4h_7d_32oc_r5.pt


## Preparação da série e definição do teste

Um dia é considerado disponível somente quando possui as 24 horas observadas. Para prever um dia central, todos os dias de sua janela de 3, 5 ou 7 dias também precisam estar completos.

- **Dia com chuva:** a GRU distribui o total diário.
- **Dia seco completo:** o notebook escreve zero diretamente, sem executar a GRU.
- **Dia com falha ou janela incompleta:** o CSV conserva `NaN`; esse caso não pode ser confundido com ausência de chuva.


In [6]:
def carregar_horario(caminho=CAMINHO_DADOS):
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(
            f"Dados horários não encontrados: {caminho.resolve()}. Execute primeiro o notebook 01."
        )
    df = pd.read_csv(caminho, parse_dates=["data_hora"])
    exigidas = {"data_hora", "precipitacao_mm"}
    if not exigidas.issubset(df.columns):
        raise ValueError(f"O arquivo deve conter as colunas {sorted(exigidas)}")
    df["precipitacao_mm"] = pd.to_numeric(df["precipitacao_mm"], errors="coerce")
    if df["data_hora"].isna().any():
        raise ValueError("Existem datas inválidas em data_hora.")
    if df["data_hora"].duplicated().any():
        raise ValueError("Existem horários duplicados no arquivo horário.")
    fora_hora = df["data_hora"] != df["data_hora"].dt.floor("h")
    if fora_hora.any():
        raise ValueError("Existem registros fora da hora cheia.")
    return df.sort_values("data_hora").set_index("data_hora")


def extrair_dias_completos(horario):
    arrays = {}
    grupos = horario["precipitacao_mm"].groupby(horario.index.normalize())
    for data, serie in grupos:
        data = pd.Timestamp(data)
        serie = serie.sort_index()
        esperado = pd.date_range(data, periods=24, freq="h")
        if len(serie) == 24 and serie.index.equals(esperado) and serie.notna().all():
            arrays[data] = serie.to_numpy(dtype=np.float32)
    if not arrays:
        raise ValueError("Nenhum dia completo com 24 horas foi encontrado.")
    return arrays


def datas_centrais_do_teste(cfg, arrays):
    janela = int(cfg["janela_dias"])
    metade = janela // 2
    corte_val = pd.Timestamp(cfg["fim_validacao"]).normalize()
    datas_ok = set(arrays)
    elegiveis = []
    for central in sorted(arrays):
        datas_janela = [
            central + pd.Timedelta(days=deslocamento)
            for deslocamento in range(-metade, metade + 1)
        ]
        if datas_janela[0] <= corte_val:
            continue
        if all(data in datas_ok for data in datas_janela):
            elegiveis.append(central)
    return pd.DatetimeIndex(elegiveis)


horario = carregar_horario()
arrays_diarios = extrair_dias_completos(horario)
totais_diarios = {data: float(valores.sum()) for data, valores in arrays_diarios.items()}

print(f"Dias completos disponíveis: {len(arrays_diarios):,}")
print(f"Período completo: {min(arrays_diarios).date()} a {max(arrays_diarios).date()}")


Dias completos disponíveis: 3,939
Período completo: 2014-12-18 a 2025-09-29


## Geração das séries


In [7]:
@torch.no_grad()
def prever_dias_chuvosos(modelo, cfg, datas, totais, device=DEVICE, batch_size=BATCH_SIZE_PREVISAO):
    datas = pd.DatetimeIndex(datas)
    n_outputs = int(cfg["n_outputs"])
    previsoes = np.zeros((len(datas), n_outputs), dtype=np.float32)
    totais_centrais = np.array([totais[d] for d in datas], dtype=np.float32)
    mascara_chuva = totais_centrais > TOLERANCIA_DIA_SECO_MM

    # Dias secos já permanecem com zero. Somente os chuvosos passam pela GRU.
    indices_chuva = np.flatnonzero(mascara_chuva)
    if len(indices_chuva) == 0:
        return previsoes, totais_centrais, mascara_chuva

    metade = int(cfg["janela_dias"]) // 2
    media = float(cfg["media_log_treino"])
    desvio = float(cfg["desvio_log_treino"])

    for inicio in range(0, len(indices_chuva), batch_size):
        idx = indices_chuva[inicio:inicio + batch_size]
        x_raw = np.stack([
            np.array([
                totais[central + pd.Timedelta(days=k)]
                for k in range(-metade, metade + 1)
            ], dtype=np.float32)[:, None]
            for central in datas[idx]
        ])
        x_norm = ((np.log1p(x_raw) - media) / desvio).astype(np.float32)
        x_tensor = torch.from_numpy(x_norm).to(device)
        total_tensor = torch.from_numpy(totais_centrais[idx]).to(device)
        pred, _, _ = modelo(x_tensor, total_tensor)
        previsoes[idx] = pred.detach().cpu().numpy().astype(np.float32)

    erro = np.abs(previsoes.sum(axis=1) - totais_centrais)
    erro_maximo = float(erro.max()) if len(erro) else 0.0
    if erro_maximo > 1e-3:
        raise RuntimeError(f"Falha na conservação diária: erro máximo de {erro.max():.6f} mm")
    return previsoes, totais_centrais, mascara_chuva


def limitar_periodo(datas, inicio=None, fim=None):
    datas = pd.DatetimeIndex(datas)
    if inicio is not None:
        datas = datas[datas >= pd.Timestamp(inicio).normalize()]
    if fim is not None:
        datas = datas[datas <= pd.Timestamp(fim).normalize()]
    return datas


def definir_periodos(selecionados, arrays, usar_comum=True):
    elegiveis = {}
    configs = {}
    for chave, caminho in selecionados.items():
        ckpt = torch_load_compativel(caminho, map_location="cpu")
        cfg = ckpt["config"]
        validar_configuracao(cfg, caminho)
        datas = datas_centrais_do_teste(cfg, arrays)
        datas = limitar_periodo(datas, DATA_INICIO_TESTE, DATA_FIM_TESTE)
        if len(datas) == 0:
            raise ValueError(f"O teste ficou vazio para {chave[0]} h / {chave[1]} dias.")
        elegiveis[chave] = datas
        configs[chave] = cfg

    if usar_comum:
        inicio_comum = max(datas.min() for datas in elegiveis.values())
        fim_comum = min(datas.max() for datas in elegiveis.values())
        if inicio_comum > fim_comum:
            raise ValueError("Não existe um período de teste comum entre os nove modelos.")
        periodos = {
            chave: datas[(datas >= inicio_comum) & (datas <= fim_comum)]
            for chave, datas in elegiveis.items()
        }
    else:
        periodos = elegiveis

    return periodos, configs


def montar_serie_continua(datas_previstas, previsoes, intervalo_horas, inicio, fim):
    inicio = pd.Timestamp(inicio).normalize()
    fim_exclusivo = pd.Timestamp(fim).normalize() + pd.Timedelta(days=1)
    grade = pd.date_range(inicio, fim_exclusivo, freq=f"{intervalo_horas}h", inclusive="left")
    serie = pd.Series(np.nan, index=grade, dtype=float, name="chuva_mm")

    for data, valores in zip(pd.DatetimeIndex(datas_previstas), previsoes):
        horarios = pd.date_range(data.normalize(), periods=len(valores), freq=f"{intervalo_horas}h")
        serie.loc[horarios] = valores.astype(float)

    saida = serie.rename_axis("data_hora").reset_index()
    return saida


def gerar_nove_series(selecionados, pasta_saida=PASTA_SAIDA):
    esperadas = {(i, j) for i in INTERVALOS_HORAS for j in JANELAS_DIAS}
    if set(selecionados) != esperadas:
        faltam = sorted(esperadas - set(selecionados))
        raise ValueError(f"A seleção deve conter as nove configurações. Faltam: {faltam}")

    periodos, configs = definir_periodos(selecionados, arrays_diarios, USAR_PERIODO_COMUM)
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)
    resumo = []

    for contador, chave in enumerate(sorted(selecionados), start=1):
        intervalo, janela = chave
        caminho = Path(selecionados[chave])
        datas = periodos[chave]
        cfg_esperada = configs[chave]
        print(f"[{contador}/9] {intervalo} h / janela {janela} dias — {caminho.name}")

        modelo, cfg = carregar_checkpoint(caminho, DEVICE)
        if int(cfg["intervalo_horas"]) != intervalo or int(cfg["janela_dias"]) != janela:
            raise ValueError(f"Configuração interna inesperada em {caminho.name}")
        if cfg != cfg_esperada:
            warnings.warn(f"A configuração de {caminho.name} mudou desde a catalogação.")

        previsoes, totais, mascara_chuva = prever_dias_chuvosos(
            modelo, cfg, datas, totais_diarios, DEVICE
        )
        serie = montar_serie_continua(
            datas_previstas=datas,
            previsoes=previsoes,
            intervalo_horas=intervalo,
            inicio=datas.min(),
            fim=datas.max(),
        )

        nome = f"serie_teste_gru_{intervalo}h_janela{janela}d.csv"
        destino = pasta_saida / nome
        serie.to_csv(destino, index=False, date_format="%Y-%m-%d %H:%M:%S", float_format="%.6f")

        erro_conservacao = np.abs(previsoes.sum(axis=1) - totais)
        resumo.append({
            "intervalo_horas": intervalo,
            "janela_dias": janela,
            "modelo": caminho.name,
            "arquivo_serie": nome,
            "inicio_teste": str(pd.Timestamp(datas.min()).date()),
            "fim_teste": str(pd.Timestamp(datas.max()).date()),
            "dias_simulaveis": int(len(datas)),
            "dias_chuvosos_desagregados": int(mascara_chuva.sum()),
            "dias_secos_preenchidos_zero": int((~mascara_chuva).sum()),
            "intervalos_no_csv": int(len(serie)),
            "intervalos_com_falha": int(serie["chuva_mm"].isna().sum()),
            "erro_conservacao_max_mm": (
                float(erro_conservacao.max()) if len(erro_conservacao) else 0.0
            ),
        })
        del modelo
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    resumo_df = pd.DataFrame(resumo).sort_values(["intervalo_horas", "janela_dias"])
    resumo_df.to_csv(pasta_saida / "resumo_series_teste.csv", index=False)
    print(f"\nNove séries criadas em: {pasta_saida.resolve()}")
    display(resumo_df)
    return resumo_df


## Opção A — gerar automaticamente

Esta célula usa os modelos resolvidos anteriormente. Quando `MODELOS_ESCOLHIDOS` contém `None`, é escolhido o menor MSE de validação disponível para aquela combinação.


In [8]:
# Descomente e execute para gerar imediatamente os nove CSVs:
resumo_series = gerar_nove_series(selecionados)


[1/9] 1 h / janela 3 dias — gru_1h_3d_32oc_r5.pt
[2/9] 1 h / janela 5 dias — gru_1h_5d_32oc_r5.pt
[3/9] 1 h / janela 7 dias — gru_1h_7d_32oc_r5.pt
[4/9] 2 h / janela 3 dias — gru_2h_3d_32oc_r5.pt
[5/9] 2 h / janela 5 dias — gru_2h_5d_32oc_r5.pt
[6/9] 2 h / janela 7 dias — gru_2h_7d_32oc_r5.pt
[7/9] 4 h / janela 3 dias — gru_4h_3d_32oc_r5.pt
[8/9] 4 h / janela 5 dias — gru_4h_5d_32oc_r5.pt
[9/9] 4 h / janela 7 dias — gru_4h_7d_32oc_r5.pt

Nove séries criadas em: B:\Codigos\DesgracaoChuvas\resultados\series_teste_desagregadas


,intervalo_horas,janela_dias,modelo,arquivo_serie,inicio_teste,fim_teste,dias_simulaveis,dias_chuvosos_desagregados,dias_secos_preenchidos_zero,intervalos_no_csv,intervalos_com_falha,erro_conservacao_max_mm
0,1,3,gru_1h_3d_32oc_r5.pt,serie_teste_gru_1h_janela3d.csv,2024-02-20,2025-09-26,585,231,354,14040,0,0.000015
1,1,5,gru_1h_5d_32oc_r5.pt,serie_teste_gru_1h_janela5d.csv,2024-02-20,2025-09-26,585,231,354,14040,0,0.000008
2,1,7,gru_1h_7d_32oc_r5.pt,serie_teste_gru_1h_janela7d.csv,2024-02-20,2025-09-26,585,231,354,14040,0,0.000008
3,2,3,gru_2h_3d_32oc_r5.pt,serie_teste_gru_2h_janela3d.csv,2024-02-20,2025-09-26,585,231,354,7020,0,0.000015
4,2,5,gru_2h_5d_32oc_r5.pt,serie_teste_gru_2h_janela5d.csv,2024-02-20,2025-09-26,585,231,354,7020,0,0.000015
5,2,7,gru_2h_7d_32oc_r5.pt,serie_teste_gru_2h_janela7d.csv,2024-02-20,2025-09-26,585,231,354,7020,0,0.000015
6,4,3,gru_4h_3d_32oc_r5.pt,serie_teste_gru_4h_janela3d.csv,2024-02-20,2025-09-26,585,231,354,3510,0,0.000008
7,4,5,gru_4h_5d_32oc_r5.pt,serie_teste_gru_4h_janela5d.csv,2024-02-20,2025-09-26,585,231,354,3510,0,0.000008
8,4,7,gru_4h_7d_32oc_r5.pt,serie_teste_gru_4h_janela7d.csv,2024-02-20,2025-09-26,585,231,354,3510,0,0.000015


## Opção B — selecionar os nove modelos por interface

Cada lista mostra apenas os checkpoints compatíveis com sua combinação de intervalo e janela. Depois de revisar as nove escolhas, clique em **Gerar 9 séries CSV**.


In [ ]:
try:
    import ipywidgets as widgets

    seletores = {}
    linhas_interface = []
    for intervalo in INTERVALOS_HORAS:
        for janela in JANELAS_DIAS:
            chave = (intervalo, janela)
            candidatos = catalogo[
                (catalogo["intervalo_horas"] == intervalo)
                & (catalogo["janela_dias"] == janela)
            ].copy()
            candidatos["ordem_mse"] = candidatos["MSE_validacao"].fillna(np.inf)
            candidatos = candidatos.sort_values(["ordem_mse", "arquivo"])
            opcoes = []
            for _, linha in candidatos.iterrows():
                mse = linha["MSE_validacao"]
                mse_txt = f"MSE val={mse:.5g}" if pd.notna(mse) else "MSE val ausente"
                rotulo = (
                    f"{linha['arquivo']} | {mse_txt} | "
                    f"ocultos={linha['hidden_size']} | camadas={linha['num_layers']}"
                )
                opcoes.append((rotulo, linha["caminho"]))
            seletor = widgets.Dropdown(
                options=opcoes,
                description=f"{intervalo}h-{janela}d:",
                layout=widgets.Layout(width="950px"),
                style={"description_width": "90px"},
            )
            seletores[chave] = seletor
            linhas_interface.append(seletor)

    botao_gerar = widgets.Button(
        description="Gerar 9 séries CSV",
        button_style="primary",
        icon="save",
        layout=widgets.Layout(width="220px"),
    )
    saida_interface = widgets.Output()

    def ao_gerar_series(_):
        with saida_interface:
            clear_output(wait=True)
            escolhas_interface = {
                chave: Path(seletor.value)
                for chave, seletor in seletores.items()
            }
            gerar_nove_series(escolhas_interface)

    botao_gerar.on_click(ao_gerar_series)
    display(widgets.VBox([
        widgets.HTML("<b>Escolha um checkpoint para cada configuração</b>"),
        *linhas_interface,
        botao_gerar,
        saida_interface,
    ]))
except ImportError:
    print("ipywidgets não está instalado. Use a Opção A.")


## Conferência dos arquivos já gerados


In [ ]:
arquivos_gerados = sorted(PASTA_SAIDA.glob("serie_teste_gru_*h_janela*d.csv"))
print(f"Séries encontradas: {len(arquivos_gerados)}")
for caminho in arquivos_gerados:
    df = pd.read_csv(caminho, parse_dates=["data_hora"])
    print(
        f"{caminho.name}: {len(df):,} linhas | "
        f"{df['data_hora'].min()} a {df['data_hora'].max()} | "
        f"falhas={df['chuva_mm'].isna().sum():,}"
    )


## Resultado esperado

A pasta `resultados/series_teste_desagregadas/` conterá:

```text
serie_teste_gru_1h_janela3d.csv
serie_teste_gru_1h_janela5d.csv
serie_teste_gru_1h_janela7d.csv
serie_teste_gru_2h_janela3d.csv
serie_teste_gru_2h_janela5d.csv
serie_teste_gru_2h_janela7d.csv
serie_teste_gru_4h_janela3d.csv
serie_teste_gru_4h_janela5d.csv
serie_teste_gru_4h_janela7d.csv
resumo_series_teste.csv
```

O próximo notebook poderá ler essas séries contínuas, calcular acumulados móveis e extrair máximas por duração. Nesta etapa, ainda não são calculados acumulados nem máximas.
